### RAG with Memory

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader('./snow-white.pdf')
documents = loader.load()

# 개행문자 제거
for doc in documents:
    doc.page_content = doc.page_content.replace('\n',' ')

documents

c:\Users\Admin\miniconda3\envs\pystudy_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2023-09-12T11:20:24+09:00', 'title': 'PowerPoint 프레젠테이션', 'author': 'PC', 'moddate': '2023-09-12T11:20:24+09:00', 'source': './snow-white.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='백설공주 옛날 어느 왕국에 공주님이 태어났어요. “어쩜 이렇게 어여쁠까? 살결이 눈처럼 하얗구나. 백 설공주라고 불러야겠다.” 왕과 왕비는 갓 태어난 딸을 보며 기뻐했어요. 하지만 기쁨도 잠시, 왕비는 곧 세상을 떠나고 말았어 요.'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2023-09-12T11:20:24+09:00', 'title': 'PowerPoint 프레젠테이션', 'author': 'PC', 'moddate': '2023-09-12T11:20:24+09:00', 'source': './snow-white.pdf', 'total_pages': 6, 'page': 1, 'page_label': '2'}, page_content='왕은 아름다운 새 왕비를 맞았어요. 그런데 새 왕비는 자기보다 아름다운 사람을 두고 보 지 못했어요. 왕비는 진실만을 말하는 요술 거울에게 늘 이렇게 물 었어요. “거울아, 거울아. 이 세상에서 누가 가장 아름답니?” “이 세상에서 가장 아름다운 사람은 왕비님입니다.” 그 대답을 들어야만 차가운 왕비 얼굴에 미소가 번졌 지요. 시간이 흘러 백설공주는 어여쁜 소녀가 되었어요

### Text Splitter 텍스트 분할
문서를 작은 청크로 분할하는 데 사용한다.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)

docs = splitter.split_documents(documents)

print(len(docs))
docs

26


[Document(metadata={'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2023-09-12T11:20:24+09:00', 'title': 'PowerPoint 프레젠테이션', 'author': 'PC', 'moddate': '2023-09-12T11:20:24+09:00', 'source': './snow-white.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='백설공주 옛날 어느 왕국에 공주님이 태어났어요. “어쩜 이렇게 어여쁠까? 살결이 눈처럼 하얗구나. 백 설공주라고 불러야겠다.” 왕과 왕비는 갓 태어난 딸을 보며 기뻐했어요. 하지만'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2023-09-12T11:20:24+09:00', 'title': 'PowerPoint 프레젠테이션', 'author': 'PC', 'moddate': '2023-09-12T11:20:24+09:00', 'source': './snow-white.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='딸을 보며 기뻐했어요. 하지만 기쁨도 잠시, 왕비는 곧 세상을 떠나고 말았어 요.'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2023-09-12T11:20:24+09:00', 'title': 'PowerPoint 프레젠테이션', 'author':

In [5]:
%pip install langchain-chroma

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_chroma.vectorstores import Chroma

# Embedding & Store
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

vector_store = Chroma.from_documents(docs, embeddings)

# 유사도를 기반으로 3개를 찾아옴
retrieval = vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k':3}
)

### RAG with History

In [8]:
# 프롬프트 생성
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ('system', 'context 기반으로만 답변해주세요. {context}'), # context는 retrieval에서 가져온 문서
    MessagesPlaceholder(variable_name='history'),
    ('human', '{query}')
])

In [13]:
# 모델 및 체인 생성
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough    # 체인 생성에 필요
from langchain_openai import ChatOpenAI
from operator import itemgetter                             # 체인 생성에 필요

output_parser = StrOutputParser()

model = ChatOpenAI(
    model='gpt-5-nano',
    temperature=0.3
)

# 필요한 것은 query(사용자 질문), history(기록), context(답변) -> RunnablePassthrough.assign() 사용
chain = (
    RunnablePassthrough.assign(
    # 윗 단계를 거치면 입력 데이터의 형태 {'query':'질문...', 'history':[대화내역들...], 'context':'답변...'}
    context=itemgetter('query') | retrieval
    )
    | prompt
    | model
    | output_parser
)

In [14]:
# 메모리 추가(RunnableWithMessageHistory) - ChatMessageHistory 이용
# 기존 chain을 감싸서, 대화 기록을 자동으로 읽고 쓰는 기능
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id):
    if session_id not in store: # store에 session_id가 존재하지 않는다면 새롭게 생성
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history, # Key와 Value가 똑같다면 이런식으로 넣어도 동작한다.
    input_messages_key='query',
    history_messages_key='history'
)


In [16]:
# 실행 (대화 테스트, 문맥이 이어지는지 테스트 해보기)
response = with_message_history.invoke(
    {'query':'백설공주는 왜 숲으로 도망갔나요?'},
    config={'configurable':{'session_id':'abc'}}
)

response

'왕비가 백설공주를 죽이라고 명령했기 때문이에요. 그러나 사냥꾼은 그럴 수 없어서 백설공주를 멀리 떠나 숲으로 도망가게 했어요.'

In [20]:
response = with_message_history.invoke(
    {'query':'내가 너한테 뭘 물어봤었지?'},
    config={'configurable':{'session_id':'abc'}}
)

response

'이전에 물어보신 내용은 "백설공주는 왜 숲으로 도망갔나요?"였어요. 이 질문을 두 번 하신 거죠.'

In [19]:
response = with_message_history.invoke(
    {'query':'첨부된 Snow White랑 원래 Snow White랑 내용 차이가 있니?'},
    config={'configurable':{'session_id':'abc'}}
)

response

'네. 첨부된 Snow White 자료와 일반적인 원전 Snow White(그림 형제 버전) 사이에 차이가 있습니다. 제공된 문서에서 보이는 내용 위주로 요약하면:\n\n- 첨부된 버전의 특징\n  - 왕비의 거울 이야기가 시작점으로 보이고, 독이 든 사과를 이용한 사건이 핵심으로 간략히 제시됩니다.\n  - 과일 장수로 변장해 백설공주에게 독이 든 사과를 건네는 장면이 포함되어 있습니다.\n  - 일곱 난쟁이의 오두막 방문 등의 사건이 언급되지만, 이후의 전개(백설공주의 부활, 왕비의 최후 등)는 자세히 다루지 않거나 생략된 구성이 보입니다.\n  - 전체가 아동용/요약 형태의 간단한 서술로 제시됩니다.\n\n- 원전(그림 형제 버전)과의 차이\n  - 원전에는 사냥꾼의 행위와 그 후속 이야기(사냥꾼이 백설공주를 죽이려 하지만 실패해 보석/심장의 증거로 거짓 증거를 가져오는 내용 포함), 독에 의해 “죽은 듯한 수면” 상태, 난쟁이들이 만드는 유리관, 왕자의 입맞춤으로 부활, 그리고 왕비의 최후 등 더 자세하고 긴 전개가 있습니다.\n  - 첨부 버전은 이 후반부 사건들을 생략하거나 축약했습니다.\n\n정리하자면, 첨부 자료는 핵심 사건(거울-독 사과-난쟁이의 오두막) 중심의 간략한 버전이고, 원전에는 더 긴 전개와 잔혹한 요소, 최후의 결말까지 포함되어 있습니다. 필요하시면 두 버전의 구체적인 발췌문을 비교해 드리겠습니다.'